In [15]:
#| default_exp io.import_data

In [16]:
#| export
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd
import xarray as xr

logger = logging.getLogger("myproj.io")

# Daten einlesen und erster Überblick

In [17]:
#| export
def find_project_root() -> Path:
    """
    Findet den Projektroot, indem vom aktuellen Arbeitsverzeichnis
    nach oben gesucht wird, bis eine pyproject.toml gefunden wird.
    """
    current = Path.cwd().resolve()
    logger.debug("Suche Projektroot ab: %s", current)

    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists():
            logger.debug("Projektroot gefunden: %s", path)
            return path

    logger.error("Projektroot nicht gefunden ab: %s", current)
    raise FileNotFoundError(
        "Projektroot nicht gefunden. Stelle sicher, dass du im Projekt "
        "oder in einem Unterordner des Projekts arbeitest."
    )


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"

In [18]:
#| export
def get_raw_file_path(filename: str) -> Path:
    """
    Gibt den vollständigen Pfad zu einer Datei im data/raw-Ordner zurück.
    """
    file_path = DATA_RAW / filename

    if not file_path.exists():
        logger.error("Rohdatei nicht gefunden: %s", file_path)
        raise FileNotFoundError(f"Datei nicht gefunden: {file_path}")

    logger.debug("Rohdatei gefunden: %s", file_path)
    return file_path

In [19]:
#| export
def load_raw_data(filename: str):
    """
    Lädt eine Datei aus data/raw abhängig von ihrer Dateiendung.

    Unterstützte Formate:
    - .xlsx, .xls -> pandas DataFrame
    - .nc -> xarray Dataset
    """
    file_path = get_raw_file_path(filename)
    suffix = file_path.suffix.lower()
    logger.info("Lade Rohdatei: %s", file_path.name)

    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(file_path)
        logger.info(
            "load_raw_data | file: %s | rows: %d | cols: %d",
            filename,
            len(df),
            len(df.columns),
        )
        return df

    if suffix == ".nc":
        ds = xr.open_dataset(file_path, engine="h5netcdf")
        logger.info(
            "load_raw_data | file: %s | variables: %d | dimensions: %s",
            filename,
            len(ds.data_vars),
            dict(ds.sizes),
        )
        return ds

    raise ValueError(
        f"Nicht unterstütztes Dateiformat: {suffix}. Unterstützt werden .xlsx, .xls, .nc"
    )

## Daten einlesen

In [20]:
emdat = load_raw_data("public_emdat_1991_2024.xlsx")
climate = load_raw_data("omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc")

## EDA für EM-DAT

In [21]:
print("EM-DAT shape:", emdat.shape)
display(emdat.head())

EM-DAT shape: (20657, 47)


,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,GADM Admin Units,Entry Date,Last Update
0,2018-0040-BRA,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4576,NaN,BRA,...,NaN,NaN,NaN,10000.0,12492.0,80.049596,"[{""adm2_code"":9961,""adm2_name"":""Rio De Janeiro""}]","[{""gid_2"":""BRA.19.68_2"",""migration_date"":""2025...",2018-02-20,2025-12-20
1,2002-0351-USA,No,nat-cli-wil-for,Natural,Climatological,Wildfire,Forest fire,NaN,NaN,USA,...,NaN,NaN,NaN,20000.0,34879.0,57.341840,"[{""adm1_code"":3219,""adm1_name"":""Colorado""}]","[{""gid_1"":""USA.6_1"",""migration_date"":""2025-12-...",2003-07-01,2025-12-20
2,1990-9604-BWA,Yes,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,BWA,...,NaN,NaN,NaN,NaN,NaN,44.731153,NaN,NaN,2006-02-13,2025-03-14
3,1990-9604-LSO,Yes,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,LSO,...,NaN,NaN,NaN,NaN,NaN,44.731153,NaN,NaN,2006-02-13,2025-03-14
4,1990-9604-MOZ,Yes,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,MOZ,...,NaN,NaN,NaN,50000.0,115164.0,43.416157,NaN,NaN,2007-05-05,2025-03-14


In [22]:
emdat.info()

<class 'pandas.DataFrame'>
RangeIndex: 20657 entries, 0 to 20656
Data columns (total 47 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   DisNo.                                     20657 non-null  str    
 1   Historic                                   20657 non-null  str    
 2   Classification Key                         20657 non-null  str    
 3   Disaster Group                             20657 non-null  str    
 4   Disaster Subgroup                          20657 non-null  str    
 5   Disaster Type                              20657 non-null  str    
 6   Disaster Subtype                           20657 non-null  str    
 7   External IDs                               4376 non-null   str    
 8   Event Name                                 6697 non-null   str    
 9   ISO                                        20657 non-null  str    
 10  Country                          

In [23]:
emdat.columns.tolist()

['DisNo.',
 'Historic',
 'Classification Key',
 'Disaster Group',
 'Disaster Subgroup',
 'Disaster Type',
 'Disaster Subtype',
 'External IDs',
 'Event Name',
 'ISO',
 'Country',
 'Subregion',
 'Region',
 'Location',
 'Origin',
 'Associated Types',
 'OFDA/BHA Response',
 'Appeal',
 'Declaration',
 "AID Contribution ('000 US$)",
 'Magnitude',
 'Magnitude Scale',
 'Latitude',
 'Longitude',
 'River Basin',
 'Start Year',
 'Start Month',
 'Start Day',
 'End Year',
 'End Month',
 'End Day',
 'Total Deaths',
 'No. Injured',
 'No. Affected',
 'No. Homeless',
 'Total Affected',
 "Reconstruction Costs ('000 US$)",
 "Reconstruction Costs, Adjusted ('000 US$)",
 "Insured Damage ('000 US$)",
 "Insured Damage, Adjusted ('000 US$)",
 "Total Damage ('000 US$)",
 "Total Damage, Adjusted ('000 US$)",
 'CPI',
 'Admin Units',
 'GADM Admin Units',
 'Entry Date',
 'Last Update']

## EDA für Climate-Dataset

In [24]:
print(climate)

<xarray.Dataset> Size: 150kB
Dimensions:                                    (time: 9405)
Coordinates:
  * time                                       (time) datetime64[ns] 75kB 199...
Data variables:
    MSL_filtered_GIA_corrected_adjusted        (time) float32 38kB ...
    trend_MSL_filtered_GIA_corrected_adjusted  (time) float32 38kB ...
Attributes:
    title:        Mediterranean Sea area Averaged Mean Sea Level from DUACS D...
    institution:  CLS
    references:   http://marine.copernicus.eu
    Conventions:  CF-1.7
    source:       The values are based on the two-satellite merged altimeter ...
    licence:      http://marine.copernicus.eu/services-portfolio/service-comm...
    area:         Mediterranean Sea
    comment:      Period : 1999-02-20 to 2024-11-19.
    credit:       E.U. Copernicus Marine Service Information
    contact:      https://marine.copernicus.eu/contact


In [25]:
print("Data variables:", list(climate.data_vars))
print("Coordinates:", list(climate.coords))
print("Dimensions:", climate.dims)

Data variables: ['MSL_filtered_GIA_corrected_adjusted', 'trend_MSL_filtered_GIA_corrected_adjusted']
Coordinates: ['time']
Dimensions: FrozenMappingWarningOnValuesAccess({'time': 9405})


In [26]:
# Umwandeln in DataFrame für eine tabellarische Ansicht
climate_df = climate.to_dataframe().reset_index()
print("Climate DataFrame shape:", climate_df.shape)
display(climate_df.head())

Climate DataFrame shape: (9405, 3)


,time,MSL_filtered_GIA_corrected_adjusted,trend_MSL_filtered_GIA_corrected_adjusted
0,1999-02-20,3.687602,2.171483
1,1999-02-21,3.684773,2.171927
2,1999-02-22,3.681582,2.172372
3,1999-02-23,3.678028,2.172817
4,1999-02-24,3.674113,2.173261


In [27]:
if "time" in climate_df.columns:
    print("Zeitspanne:", climate_df["time"].min(), "bis", climate_df["time"].max())
    print("Datentyp von time:", climate_df["time"].dtype)

Zeitspanne: 1999-02-20 00:00:00 bis 2024-11-19 00:00:00
Datentyp von time: datetime64[ns]
